# ⚡ Transpilación de 1D-CNN + LSTM a TFLite Micro INT8 y C++ para ESP32-S3

Este cuaderno toma el modelo Keras entrenado `myotensor_proto_net_lstm_trained.keras` y:
1. Aplica cuantización entera completa **INT8 (Full Integer Quantization)** con dataset representativo.
2. Valida la integridad del archivo `.tflite` y sus operadores.
3. Genera automáticamente los archivos C++ directamente en [firmware/Classifier/](file:///home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/):
   - `NN_model.h` (Cabecera)
   - `NN_model.cpp` (Arreglo con `alignas(16)` para SIMD en ESP32-S3)
   - `cnn_scaler_params.h` (Parámetros de normalización Z-score).


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Forzar CPU para conversiones limpias
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import tensorflow as tf
import joblib
from pathlib import Path

def load_env_variables():
    start_dir = Path(os.getcwd())
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()
models_dir = os.environ["MODELS_DL_PROTO"]
base_path = os.environ["PROCESSED_TENSOR_PROTO"]
classifier_root = Path(os.environ["PROJECT_ROOT"]) / "firmware" / "Classifier" 


In [ ]:
# 📥 Cargar modelo entrenado y datos para calibración de cuantización
model_path = os.path.join(models_dir, "myotensor_proto_net_lstm_trained.keras")
print(f"📥 Cargando modelo Keras desde: {model_path}")
model = tf.keras.models.load_model(model_path)
model.summary()

train_data_path = os.path.join(base_path, "X_train.npy")
print(f"📖 Cargando X_train desde: {train_data_path}")
X_train = np.load(train_data_path)

# Generador de dataset representativo para cuantización INT8
def representative_dataset():
    np.random.seed(42)
    indices = np.random.choice(X_train.shape[0], size=min(250, X_train.shape[0]), replace=False)
    for idx in indices:
        sample = X_train[idx].astype(np.float32)
        if sample.ndim == 1:
            sample = np.expand_dims(sample, axis=-1)
        yield [np.expand_dims(sample, axis=0)]

print("✅ Generador representative_dataset configurado.")


In [ ]:
# ⚙️ Configurar conversor TFLite para cuantización INT8 completa
print("⚙️ Convirtiendo modelo Keras a TFLite INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print(f"✅ Conversión INT8 exitosa! Tamaño: {len(tflite_model):,} bytes ({len(tflite_model)/1024:.2f} KB)")

tflite_save_path = os.path.join(models_dir, "myotensor_proto_net_lstm.tflite")
with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)
print(f"💾 Archivo TFLite guardado en: {tflite_save_path}")


In [ ]:
# 🔍 Validar modelo cuantizado con el Intérprete TFLite
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print(f"📥 Input TFLite: shape={input_details['shape']}, dtype={input_details['dtype']}, scale={input_details['quantization'][0]}, zero_point={input_details['quantization'][1]}")
print(f"📤 Output TFLite: shape={output_details['shape']}, dtype={output_details['dtype']}")

# Prueba de inferencia rápida
test_sample = (X_train[0:1] / input_details['quantization'][0] + input_details['quantization'][1]).astype(np.int8)
interpreter.set_tensor(input_details['index'], test_sample)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details['index'])
print(f"🎯 Inferencia de prueba exitosa! Salida cuantizada: {output_data}")


In [ ]:
# 📝 Escribir archivos C++ (Header y Source) directamente en firmware/Classifier
header_path = classifier_root / "include" / "NN_model.h"
source_path = classifier_root / "src" / "NN_model.cpp"
scaler_path = classifier_root / "include" / "cnn_scaler_params.h"

# 1. Escribir Header (.h)
header_content = """#ifndef NN_MODEL_H
#define NN_MODEL_H

// Declaración externa del modelo cuantizado TFLite Micro (CNN-LSTM)
extern const unsigned char g_model_data[];
extern const int g_model_data_len;

#endif // NN_MODEL_H
"""
with open(header_path, "w", encoding="utf-8") as f:
    f.write(header_content)
print(f"✅ Archivo de cabecera creado: {header_path}")

# 2. Formatear bytes a hexadecimal en C++
c_hex_array = [f"0x{val:02X}" for val in tflite_model]
formatted_array = ""
for i in range(0, len(c_hex_array), 12):
    formatted_array += "  " + ", ".join(c_hex_array[i:i+12]) + ",\n"
formatted_array = formatted_array.rstrip(",\n") + "\n"

# 3. Escribir Source (.cpp) con alignas(16) para SIMD en ESP32-S3
source_content = f"""#include \"NN_model.h\"

// Alineación requerida por TensorFlow Lite Micro y SIMD en ESP32-S3
alignas(16) const unsigned char g_model_data[] = {{
{formatted_array}}};

const int g_model_data_len = {len(tflite_model)};
"""
with open(source_path, "w", encoding="utf-8") as f:
    f.write(source_content)
print(f"✅ Archivo de definición creado: {source_path}")

# 4. Generar cnn_scaler_params.h desde std_scaler.bin
std_scaler_file = Path(base_path) / "std_scaler.bin"
if std_scaler_file.exists():
    scaler = joblib.load(std_scaler_file)
    signal_mean = float(scaler.mean_[0])
    signal_std = float(scaler.scale_[0])
else:
    signal_mean = 0.0
    signal_std = 1.0

cnn_scaler_content = f"""#ifndef CNN_SCALER_PARAMS_H
#define CNN_SCALER_PARAMS_H

// Parámetros del StandardScaler para la señal cruda autogenerados desde Python
const float signal_mean = {signal_mean:.9f}f;
const float signal_std = {signal_std:.9f}f;

#endif // CNN_SCALER_PARAMS_H
"""
with open(scaler_path, "w", encoding="utf-8") as f:
    f.write(cnn_scaler_content)
print(f"✅ Parámetros de escala generados: {scaler_path} (mean={signal_mean:.6f}, std={signal_std:.6f})")


## 🚀 Listo para Flashear al ESP32-S3
Para compilar y subir el firmware al microcontrolador:
```bash
cd /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier
pio run -t upload
```
